[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/20.%20The%20SLAP%20for%20Out-of-Gauge%20%28OOG%29%20Containers%20Problem/P20-Tier-4_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---

# 20. The SLAP for Out-of-Gauge (OOG) Containers Problem

## Tier 4: Few-Shot Learning for OOG Container Assignment

### Purpose and scope

This notebook builds a small neural network that scores container–slot pairs, adapts its parameters using a small support set, and evaluates pair-label predictions on a separate query set. It also turns the scores into a greedy, constraint-checked assignment. The example illustrates the mechanics of few-shot adaptation; it does not establish that the model generalizes to unseen vessel configurations.

The input is a small, fixed instance containing three OOG containers, four vessel slots, a cost matrix, and a stability matrix. The output includes pair-classification metrics, a greedy assignment, and a cost-only assignment reference. A second vessel configuration illustrates adaptation when slot features change.

### Modeling notes

- Pair labels are proxy labels derived from the supplied cost and stability values. They are not labels from an optimal assignment.
- The network starts from random weights and is updated using the support examples. There is no outer-loop meta-training across multiple tasks, so this code demonstrates support-set adaptation rather than a trained meta-learning system.
- The assignment rule selects the highest-scoring feasible unused slot for each container, considering containers with fewer feasible slots first. This is a greedy rule, not a global optimality guarantee.
- Feasibility checks use the supplied weight capacities, height limits, and bay weight limits. Other operational requirements—such as lashing compatibility, vessel-wide stability, and crane scheduling—are outside this instance's model.
- The cost-only reference minimizes the supplied cost matrix; it does not enforce the physical feasibility checks used by the greedy assignment.
- NumPy's random number generator is not seeded in this example. Support-set selection and initial network weights can therefore vary between runs.

### Learning setup

For each container \(i\) and slot \(j\), the feature encoder produces \(x_{ij}\in\mathbb{R}^{156}\). A neural network maps these features to a score \(p_{ij}\in(0,1)\), interpreted as the predicted probability of a positive proxy label. The support set is used for gradient-based adaptation; the query set is held out from that adaptation and used for evaluation.

For a support set \(S\), the binary cross-entropy objective is

$$
L_S(\theta)=-\frac{1}{|S|}\sum_{(x,y)\in S}\left[y\log p_{\theta}(x)+(1-y)\log\left(1-p_{\theta}(x)\right)\right].
$$

One adaptation step is \(\theta\leftarrow\theta-\alpha\nabla_{\theta}L_S(\theta)\), where \(\theta\) contains the network weights and biases and \(\alpha\) is the inner-loop learning rate. Repeating this update for the specified number of steps produces adapted parameters. This is ordinary gradient-based fine-tuning on a small sample; a meta-learning claim would require training across a collection of tasks, which is not part of this example.

The assignment stage uses these adapted scores but also checks operational constraints. It is separate from the pair-classification evaluation: high pair-label accuracy does not by itself imply a low-cost or globally optimal assignment.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from scipy.optimize import linear_sum_assignment

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

print("Libraries imported successfully.")

In [ ]:
@dataclass
class OOGContainer:
    """Physical and operational attributes of an OOG container."""

    id: int
    weight: float  # tons
    height: float  # meters
    length: float  # meters
    container_type: str
    lashing_coeff: float
    value_category: int
    handling_priority: float
    destination_urgency: float


@dataclass
class VesselSlot:
    """Capacity, location, and access attributes of a vessel slot."""

    id: int
    weight_capacity: float  # tons
    height_limit: float  # meters
    bay_id: int
    crane_distance: float
    slot_position: str
    accessibility_score: float
    stability_factor: float


@dataclass
class FeatureVector:
    """Encoded container, slot, interaction, and combined features."""

    container_features: np.ndarray
    slot_features: np.ndarray
    interaction_features: np.ndarray
    combined_features: np.ndarray


@dataclass
class FSLProblem:
    """Data required for the few-shot assignment example."""

    containers: List[OOGContainer]
    slots: List[VesselSlot]
    cost_matrix: np.ndarray
    stability_matrix: np.ndarray
    bay_weight_limits: Dict[int, float]
    feature_dim: int


print("Problem data structures are ready.")

In [ ]:
containers = [
    OOGContainer(1, 45, 2.8, 12.0, "flat rack", 2.1, 3, 0.7, 0.8),
    OOGContainer(2, 32, 4.2, 6.0, "open-top", 3.8, 2, 0.9, 0.6),
    OOGContainer(3, 38, 2.6, 12.0, "platform", 1.9, 3, 0.5, 0.4),
]

slots = [
    VesselSlot(1, 50, 3.0, 2, 15.0, "forward", 0.8, 0.9),
    VesselSlot(2, 40, 5.0, 3, 12.0, "midship", 0.9, 0.7),
    VesselSlot(3, 45, 3.0, 3, 18.0, "midship", 0.7, 0.8),
    VesselSlot(4, 60, 4.0, 3, 10.0, "aft", 0.95, 0.85),
]

cost_matrix = np.array(
    [
        [100, 150, 120, 90],
        [80, 70, 110, 85],
        [95, 140, 100, 80],
    ],
    dtype=float,
)

stability_matrix = np.array(
    [
        [0.8, 0.7, 0.9, 0.85],
        [0.75, 0.8, 0.7, 0.9],
        [0.85, 0.65, 0.8, 0.75],
    ],
    dtype=float,
)

bay_weight_limits = {2: 80, 3: 120}
feature_dim = 156

problem = FSLProblem(
    containers=containers,
    slots=slots,
    cost_matrix=cost_matrix,
    stability_matrix=stability_matrix,
    bay_weight_limits=bay_weight_limits,
    feature_dim=feature_dim,
)

assert cost_matrix.shape == (len(containers), len(slots))
assert stability_matrix.shape == cost_matrix.shape

print(f"Problem instance: {len(containers)} containers and {len(slots)} slots")
print(f"Combined feature dimension: {problem.feature_dim}")
for container in problem.containers:
    print(
        f"Container {container.id}: {container.container_type}, "
        f"{container.weight} tons, {container.height} meters; "
        f"value category {container.value_category}, "
        f"priority {container.handling_priority}, "
        f"urgency {container.destination_urgency}"
    )

## Feature engineering

Feature engineering converts descriptive attributes into numerical arrays that the network can process. Container and slot features are normalized using fixed scale factors, categorical attributes are represented by indicator values, and degree-two polynomial terms add simple nonlinear combinations. Pair interactions then describe element-wise agreement, absolute differences, and cosine similarity.

The final representation allocates 52 values to container attributes, 52 to slot attributes, and 52 to their interaction features, giving \(52+52+52=156\) values per pair. The interaction vector is truncated to its first 52 entries to preserve the stated input dimension. These engineered features are an input representation, not evidence that the network has learned a useful pattern.

In [ ]:
def _fit_feature_dimension(features: np.ndarray, target_dim: int) -> np.ndarray:
    """Pad or truncate an engineered vector to a fixed input size."""
    if len(features) < target_dim:
        return np.pad(features, (0, target_dim - len(features)), mode="constant")
    return features[:target_dim]


def encode_container_features(container: OOGContainer) -> np.ndarray:
    """Encode normalized physical and operational container attributes."""
    physical_features = np.array(
        [
            container.weight / 100.0,
            container.height / 10.0,
            container.length / 20.0,
            float(container.container_type == "flat rack"),
            float(container.container_type == "open-top"),
            float(container.container_type == "platform"),
            container.lashing_coeff / 5.0,
            container.value_category / 3.0,
            container.handling_priority,
            container.destination_urgency,
        ],
        dtype=float,
    )

    polynomial_features = [
        physical_features[i] * physical_features[j]
        for i in range(len(physical_features))
        for j in range(i, len(physical_features))
    ]
    trigonometric_features = [
        value
        for feature in physical_features[:5]
        for value in (np.sin(feature * np.pi), np.cos(feature * np.pi))
    ]
    all_features = np.concatenate(
        [physical_features, polynomial_features, trigonometric_features]
    )
    return _fit_feature_dimension(all_features, 52)


def encode_slot_features(slot: VesselSlot) -> np.ndarray:
    """Encode normalized capacity, location, access, and stability attributes."""
    basic_features = np.array(
        [
            slot.weight_capacity / 100.0,
            slot.height_limit / 10.0,
            slot.bay_id / 10.0,
            slot.crane_distance / 50.0,
            float(slot.slot_position == "forward"),
            float(slot.slot_position == "midship"),
            float(slot.slot_position == "aft"),
            slot.accessibility_score,
            slot.stability_factor,
            slot.id / 10.0,
        ],
        dtype=float,
    )

    polynomial_features = [
        basic_features[i] * basic_features[j]
        for i in range(len(basic_features))
        for j in range(i, len(basic_features))
    ]
    trigonometric_features = [
        value
        for feature in basic_features[:5]
        for value in (np.sin(feature * np.pi), np.cos(feature * np.pi))
    ]
    all_features = np.concatenate(
        [basic_features, polynomial_features, trigonometric_features]
    )
    return _fit_feature_dimension(all_features, 52)


def create_interaction_features(
    container_features: np.ndarray, slot_features: np.ndarray
) -> np.ndarray:
    """Build pair features from element-wise products, differences, and similarity."""
    if container_features.shape != slot_features.shape:
        raise ValueError("Container and slot feature vectors must have equal shapes.")

    elementwise_product = container_features * slot_features
    absolute_difference = np.abs(container_features - slot_features)
    norm_product = np.linalg.norm(container_features) * np.linalg.norm(slot_features)
    cosine_similarity = np.dot(container_features, slot_features) / (norm_product + 1e-8)

    return np.concatenate(
        [elementwise_product, absolute_difference, np.array([cosine_similarity])]
    )


print("Feature encoding functions are ready.")

In [ ]:
def create_feature_vector(
    problem: FSLProblem, container_id: int, slot_id: int
) -> FeatureVector:
    """Create the fixed-length feature representation for one indexed pair."""
    container_features = encode_container_features(problem.containers[container_id])
    slot_features = encode_slot_features(problem.slots[slot_id])
    interaction_features = create_interaction_features(
        container_features, slot_features
    )
    interaction_features = _fit_feature_dimension(interaction_features, 52)
    combined_features = np.concatenate(
        [container_features, slot_features, interaction_features]
    )

    if combined_features.shape != (problem.feature_dim,):
        raise ValueError(
            f"Expected {problem.feature_dim} features, "
            f"received {len(combined_features)}."
        )

    return FeatureVector(
        container_features=container_features,
        slot_features=slot_features,
        interaction_features=interaction_features,
        combined_features=combined_features,
    )


def create_feature_matrix(problem: FSLProblem) -> Tuple[np.ndarray, np.ndarray]:
    """Build pair features and cost–stability proxy labels for the instance."""
    n_containers = len(problem.containers)
    n_slots = len(problem.slots)
    expected_shape = (n_containers, n_slots)
    if problem.cost_matrix.shape != expected_shape:
        raise ValueError("The cost matrix shape must match the container and slot counts.")
    if problem.stability_matrix.shape != expected_shape:
        raise ValueError("The stability matrix shape must match the container and slot counts.")

    feature_rows = []
    pair_scores = []
    for container_id in range(n_containers):
        for slot_id in range(n_slots):
            feature_rows.append(
                create_feature_vector(problem, container_id, slot_id).combined_features
            )
            pair_scores.append(
                -problem.cost_matrix[container_id, slot_id]
                + 100.0 * problem.stability_matrix[container_id, slot_id]
            )

    feature_matrix = np.asarray(feature_rows, dtype=float)
    pair_scores = np.asarray(pair_scores, dtype=float)
    threshold = np.percentile(pair_scores, 75)
    labels = (pair_scores > threshold).astype(int)
    return feature_matrix, labels


print("Feature matrix and proxy-label functions are ready.")

## Neural network and support-set adaptation

A forward pass applies an affine transformation and a ReLU activation at each hidden layer, followed by a sigmoid output. The sigmoid maps the final value to \((0,1)\), making it suitable as a binary-label probability. During adaptation, backpropagation computes the gradient of binary cross-entropy with respect to every weight and bias; gradient descent then updates copied parameters while leaving the model's initial parameters unchanged.

For hidden layer \(\ell\), the calculation is \(a^{(\ell)}=\operatorname{ReLU}(a^{(\ell-1)}W^{(\ell)}+b^{(\ell)})\). The output is \(p=\operatorname{sigmoid}(a^{(L)}W^{(\mathrm{out})}+b^{(\mathrm{out})})\). The backward pass applies the chain rule from the output toward the input. Its computational work per step is dominated by matrix multiplications and scales with the number of examples and the layer widths.

In [ ]:
class FewShotLearningModel:
    """Feed-forward binary classifier with gradient-based support adaptation."""

    def __init__(self, input_dim: int, hidden_dims: Optional[List[int]] = None):
        if input_dim <= 0:
            raise ValueError("input_dim must be positive.")
        self.input_dim = input_dim
        self.hidden_dims = hidden_dims or [128, 64, 32]
        if not self.hidden_dims or any(width <= 0 for width in self.hidden_dims):
            raise ValueError("hidden_dims must contain positive layer widths.")

        layer_dims = [input_dim] + self.hidden_dims
        self.weights: Dict[str, np.ndarray] = {}
        self.biases: Dict[str, np.ndarray] = {}
        for layer_index in range(1, len(layer_dims)):
            self.weights[f"W{layer_index}"] = (
                np.random.randn(layer_dims[layer_index - 1], layer_dims[layer_index])
                * 0.1
            )
            self.biases[f"b{layer_index}"] = np.zeros(layer_dims[layer_index])

        self.weights["W_out"] = np.random.randn(self.hidden_dims[-1], 1) * 0.1
        self.biases["b_out"] = np.zeros(1)
        self.alpha = 0.01

    @staticmethod
    def relu(values: np.ndarray) -> np.ndarray:
        return np.maximum(0.0, values)

    @staticmethod
    def sigmoid(values: np.ndarray) -> np.ndarray:
        clipped_values = np.clip(values, -500.0, 500.0)
        return 1.0 / (1.0 + np.exp(-clipped_values))

    @staticmethod
    def binary_cross_entropy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
        probabilities = np.clip(y_pred, 1e-7, 1.0 - 1e-7)
        return float(
            -np.mean(
                y_true * np.log(probabilities)
                + (1.0 - y_true) * np.log(1.0 - probabilities)
            )
        )

    def forward_with_params(
        self,
        X: np.ndarray,
        weights: Dict[str, np.ndarray],
        biases: Dict[str, np.ndarray],
    ) -> Tuple[np.ndarray, Dict[str, np.ndarray]]:
        """Return probabilities and activations for the supplied parameters."""
        if X.ndim != 2 or X.shape[1] != self.input_dim:
            raise ValueError("X must be a 2D array with the configured input dimension.")

        cache: Dict[str, np.ndarray] = {"A0": X}
        for layer_index in range(1, len(self.hidden_dims) + 1):
            z_values = (
                cache[f"A{layer_index - 1}"] @ weights[f"W{layer_index}"]
                + biases[f"b{layer_index}"]
            )
            cache[f"Z{layer_index}"] = z_values
            cache[f"A{layer_index}"] = self.relu(z_values)

        output_logits = (
            cache[f"A{len(self.hidden_dims)}"] @ weights["W_out"]
            + biases["b_out"]
        )
        predictions = self.sigmoid(output_logits)
        cache["output_logits"] = output_logits
        cache["predictions"] = predictions
        return predictions, cache

    def forward(self, X: np.ndarray) -> Tuple[np.ndarray, Dict[str, np.ndarray]]:
        """Run a forward pass using the model's current parameters."""
        return self.forward_with_params(X, self.weights, self.biases)

    def compute_gradients(
        self,
        X: np.ndarray,
        y: np.ndarray,
        output: np.ndarray,
        cache: Dict[str, np.ndarray],
        weights: Dict[str, np.ndarray],
        biases: Dict[str, np.ndarray],
    ) -> Dict[str, np.ndarray]:
        """Backpropagate binary cross-entropy gradients through all layers."""
        del biases  # Bias values are not needed to compute their gradients.
        sample_count = X.shape[0]
        if sample_count == 0:
            raise ValueError("At least one training example is required.")

        gradients: Dict[str, np.ndarray] = {}
        delta = (output - y.reshape(-1, 1)) / sample_count
        last_hidden = len(self.hidden_dims)
        gradients["dW_out"] = cache[f"A{last_hidden}"].T @ delta
        gradients["db_out"] = np.sum(delta, axis=0)
        delta = delta @ weights["W_out"].T

        for layer_index in range(last_hidden, 0, -1):
            delta = delta * (cache[f"Z{layer_index}"] > 0.0)
            gradients[f"dW{layer_index}"] = (
                cache[f"A{layer_index - 1}"].T @ delta
            )
            gradients[f"db{layer_index}"] = np.sum(delta, axis=0)
            if layer_index > 1:
                delta = delta @ weights[f"W{layer_index}"].T

        return gradients

    def adapt_to_support_set(
        self, support_X: np.ndarray, support_y: np.ndarray, num_steps: int = 5
    ) -> Dict[str, np.ndarray]:
        """Return copied parameters updated by gradient descent on support examples."""
        if support_X.shape[0] != len(support_y) or support_X.shape[0] == 0:
            raise ValueError("The support features and labels must have equal, nonzero length.")
        if num_steps < 0:
            raise ValueError("num_steps must be non-negative.")

        adapted_weights = {key: value.copy() for key, value in self.weights.items()}
        adapted_biases = {key: value.copy() for key, value in self.biases.items()}

        for _ in range(num_steps):
            output, cache = self.forward_with_params(
                support_X, adapted_weights, adapted_biases
            )
            gradients = self.compute_gradients(
                support_X,
                support_y,
                output,
                cache,
                adapted_weights,
                adapted_biases,
            )
            for key in adapted_weights:
                adapted_weights[key] -= self.alpha * gradients[f"d{key}"]
            for key in adapted_biases:
                adapted_biases[key] -= self.alpha * gradients[f"d{key}"]

        return {**adapted_weights, **adapted_biases}


print("The neural network and support adaptation are ready.")

## Support, query, and evaluation sets

A support set contains examples used to update the model; a query set contains different examples used to assess the adapted model. Keeping the sets separate prevents the reported query metrics from being calculated on the same examples used for the adaptation in that run. Here the data are container–slot pairs from one small instance, not independent vessel-level tasks, so the evaluation is illustrative and should not be interpreted as a measure of deployment performance.

In [ ]:
def create_support_and_query_sets(
    problem: FSLProblem, support_size: int = 2
) -> Tuple[Dict[str, np.ndarray], Dict[str, np.ndarray]]:
    """Randomly split pair examples into disjoint support and query sets."""
    X, y = create_feature_matrix(problem)
    if support_size <= 0 or support_size >= len(X):
        raise ValueError("support_size must be positive and smaller than the dataset.")

    support_indices = np.random.choice(len(X), support_size, replace=False)
    query_indices = np.setdiff1d(np.arange(len(X)), support_indices)
    n_slots = len(problem.slots)

    support_set = {
        "X": X[support_indices],
        "y": y[support_indices],
        "container_ids": support_indices // n_slots,
        "slot_ids": support_indices % n_slots,
    }
    query_set = {
        "X": X[query_indices],
        "y": y[query_indices],
        "container_ids": query_indices // n_slots,
        "slot_ids": query_indices % n_slots,
    }
    return support_set, query_set


def evaluate_adapted_model(
    model: FewShotLearningModel,
    adapted_params: Dict[str, np.ndarray],
    query_set: Dict[str, np.ndarray],
) -> Dict[str, float]:
    """Calculate binary classification metrics on query examples."""
    weights = {key: value for key, value in adapted_params.items() if key.startswith("W")}
    biases = {key: value for key, value in adapted_params.items() if key.startswith("b")}
    predictions, _ = model.forward_with_params(query_set["X"], weights, biases)
    binary_predictions = (predictions.ravel() >= 0.5).astype(int)
    y_true = query_set["y"].astype(int)

    true_positives = np.sum((binary_predictions == 1) & (y_true == 1))
    predicted_positives = np.sum(binary_predictions == 1)
    actual_positives = np.sum(y_true == 1)
    precision = (
        true_positives / predicted_positives if predicted_positives else 0.0
    )
    recall = true_positives / actual_positives if actual_positives else 0.0
    f1_score = (
        2.0 * precision * recall / (precision + recall)
        if precision + recall
        else 0.0
    )

    return {
        "accuracy": float(np.mean(binary_predictions == y_true)),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1_score),
    }


def generate_assignment_from_predictions(
    model: FewShotLearningModel,
    adapted_params: Dict[str, np.ndarray],
    problem: FSLProblem,
) -> np.ndarray:
    """Greedily assign containers to high-scoring feasible, unused slots."""
    weights = {key: value for key, value in adapted_params.items() if key.startswith("W")}
    biases = {key: value for key, value in adapted_params.items() if key.startswith("b")}
    n_containers = len(problem.containers)
    n_slots = len(problem.slots)

    scores = np.empty((n_containers, n_slots), dtype=float)
    for container_index in range(n_containers):
        for slot_index in range(n_slots):
            features = create_feature_vector(
                problem, container_index, slot_index
            ).combined_features.reshape(1, -1)
            prediction, _ = model.forward_with_params(features, weights, biases)
            scores[container_index, slot_index] = prediction[0, 0]

    feasible_slots = {}
    for container_index, container in enumerate(problem.containers):
        feasible_slots[container_index] = [
            slot_index
            for slot_index, slot in enumerate(problem.slots)
            if container.weight <= slot.weight_capacity
            and container.height <= slot.height_limit
            and container.weight <= problem.bay_weight_limits.get(slot.bay_id, np.inf)
        ]
        if not feasible_slots[container_index]:
            raise ValueError(
                f"Container {container.id} has no individually feasible slot."
            )

    # Constrained containers are considered first so that scarce slot options remain available.
    container_order = sorted(
        range(n_containers), key=lambda index: (len(feasible_slots[index]), index)
    )
    assignment = np.zeros((n_containers, n_slots), dtype=int)
    used_slots = set()
    bay_loads: Dict[int, float] = {}

    for container_index in container_order:
        container = problem.containers[container_index]
        candidates = sorted(
            feasible_slots[container_index],
            key=lambda slot_index: scores[container_index, slot_index],
            reverse=True,
        )
        selected_slot = None
        for slot_index in candidates:
            if slot_index in used_slots:
                continue
            slot = problem.slots[slot_index]
            proposed_bay_load = bay_loads.get(slot.bay_id, 0.0) + container.weight
            bay_limit = problem.bay_weight_limits.get(slot.bay_id, np.inf)
            if proposed_bay_load <= bay_limit:
                selected_slot = slot_index
                break

        if selected_slot is None:
            raise ValueError(
                "The greedy scoring rule could not complete a feasible assignment."
            )

        assignment[container_index, selected_slot] = 1
        used_slots.add(selected_slot)
        selected_bay = problem.slots[selected_slot].bay_id
        bay_loads[selected_bay] = bay_loads.get(selected_bay, 0.0) + container.weight

    return assignment


print("Support/query creation, evaluation, and assignment functions are ready.")

## Adaptation and assignment

The next cell follows the learning sequence: create the model, draw two support examples, adapt for five gradient steps, and evaluate on the disjoint query examples. It then uses the adapted model's pair scores in the constrained greedy assignment. The printed classification accuracy measures agreement with the proxy pair labels; it is not the percentage of containers placed in an optimal slot.

In [ ]:
print("FEW-SHOT ADAPTATION FOR OOG CONTAINER–SLOT PAIRS")
print("=" * 52)
print(f"Network: input {problem.feature_dim}, hidden layers [128, 64, 32], output 1")

model = FewShotLearningModel(
    input_dim=problem.feature_dim, hidden_dims=[128, 64, 32]
)
support_set, query_set = create_support_and_query_sets(problem, support_size=2)

print(f"Support examples: {len(support_set['X'])}")
for example_index, label in enumerate(support_set["y"]):
    container_index = support_set["container_ids"][example_index]
    slot_index = support_set["slot_ids"][example_index]
    container_id = problem.containers[container_index].id
    slot_id = problem.slots[slot_index].id
    print(
        f"  Container {container_id} to slot {slot_id}: "
        f"proxy label {label}"
    )

adapted_params = model.adapt_to_support_set(
    support_set["X"], support_set["y"], num_steps=5
)
metrics = evaluate_adapted_model(model, adapted_params, query_set)

print(f"\nQuery examples: {len(query_set['X'])}")
print("Query pair-label metrics:")
for metric_name, metric_value in metrics.items():
    print(f"  {metric_name.replace('_', ' ').title()}: {metric_value:.3f}")

fsl_assignment = generate_assignment_from_predictions(
    model, adapted_params, problem
)
fsl_cost = float(np.sum(problem.cost_matrix * fsl_assignment))
fsl_stability = float(np.sum(problem.stability_matrix * fsl_assignment))

print("\nGreedy feasible assignment:")
for container_index, container in enumerate(problem.containers):
    slot_index = int(np.flatnonzero(fsl_assignment[container_index])[0])
    slot = problem.slots[slot_index]
    print(
        f"  Container {container.id} ({container.container_type}, "
        f"{container.weight} tons, {container.height} meters) -> "
        f"slot {slot.id} (capacity {slot.weight_capacity} tons, "
        f"height limit {slot.height_limit} meters, bay {slot.bay_id}); "
        f"cost {problem.cost_matrix[container_index, slot_index]:.0f}, "
        f"stability {problem.stability_matrix[container_index, slot_index]:.2f}"
    )

print(f"Total assignment cost: {fsl_cost:.2f} units")
print(f"Total stability score: {fsl_stability:.2f}")

assert np.all(fsl_assignment.sum(axis=1) == 1)
assert np.all(fsl_assignment.sum(axis=0) <= 1)
assert np.isclose(fsl_cost, np.sum(problem.cost_matrix * fsl_assignment))

## Cost-only reference

This reference applies the Hungarian algorithm to the supplied cost matrix alone. It provides a cost benchmark for the small matrix, but it does not include height, weight-capacity, or bay-load constraints. Therefore, its assignment is not necessarily physically feasible and should not be treated as an operational solution. The comparison is about total matrix cost, not a comparison of the learning algorithm's generalization quality.

In [ ]:
row_indices, column_indices = linear_sum_assignment(problem.cost_matrix)
reference_assignment = np.zeros_like(problem.cost_matrix, dtype=int)
reference_assignment[row_indices, column_indices] = 1
reference_cost = float(np.sum(problem.cost_matrix * reference_assignment))
reference_stability = float(
    np.sum(problem.stability_matrix * reference_assignment)
)

print("COST-ONLY MATRIX REFERENCE")
print("=" * 30)
print(f"Reference cost: {reference_cost:.2f} units")
print(f"Reference stability score: {reference_stability:.2f}")
print(f"Greedy feasible assignment cost: {fsl_cost:.2f} units")
print(f"Greedy feasible assignment stability: {fsl_stability:.2f}")

cost_gap = (fsl_cost / reference_cost - 1.0) * 100.0 if reference_cost else np.nan
print(f"Cost difference relative to the matrix reference: {cost_gap:+.1f}%")
print(
    "Interpret the cost difference with care: the reference minimizes cost alone, "
    "whereas the greedy assignment enforces the modeled feasibility limits."
)

## Visual diagnostics

The first panel shows two input features, not a dimensionality-reduction result: normalized container weight and height. The second panel reports query pair-label metrics. The third compares cost and stability scores for the two assignments, which have different feasibility guarantees. The final panel repeats disjoint support/query splits for several support sizes; each point is one random split, not an averaged learning curve. Because this tiny dataset is reused and the model is randomly initialized, the plot is descriptive rather than a reliable estimate of learning efficiency.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.suptitle("Few-Shot Pair Classification and Assignment Diagnostics", fontsize=15)

ax1, ax2, ax3, ax4 = axes.ravel()
query_features = query_set["X"]
query_labels = query_set["y"]
for label, color, label_name in [(0, "tab:blue", "Negative proxy label"), (1, "tab:red", "Positive proxy label")]:
    mask = query_labels == label
    ax1.scatter(
        query_features[mask, 0],
        query_features[mask, 1],
        color=color,
        alpha=0.75,
        label=label_name,
    )
ax1.set_xlabel("Normalized container weight")
ax1.set_ylabel("Normalized container height")
ax1.set_title("Query examples by proxy label")
ax1.legend()
ax1.grid(True, alpha=0.3)

metric_names = ["Accuracy", "Precision", "Recall", "F1-score"]
metric_values = [metrics[name] for name in ["accuracy", "precision", "recall", "f1_score"]]
bars = ax2.bar(metric_names, np.array(metric_values) * 100.0, color="steelblue", alpha=0.85)
ax2.set_ylabel("Query pair-label metric (%)")
ax2.set_title("Adapted model metrics")
ax2.set_ylim(0, 100)
ax2.tick_params(axis="x", rotation=15)
ax2.grid(axis="y", alpha=0.3)
for bar, value in zip(bars, metric_values):
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f"{value * 100:.1f}%",
        ha="center",
        va="bottom",
    )

method_names = ["Cost-only reference", "Greedy feasible"]
cost_values = [reference_cost, fsl_cost]
stability_values = [reference_stability, fsl_stability]
x_positions = np.arange(len(method_names))
bar_width = 0.35
ax3.bar(
    x_positions - bar_width / 2,
    cost_values,
    bar_width,
    label="Cost (units)",
    color="steelblue",
)
ax3_twin = ax3.twinx()
ax3_twin.bar(
    x_positions + bar_width / 2,
    stability_values,
    bar_width,
    label="Stability score",
    color="seagreen",
)
ax3.set_xticks(x_positions)
ax3.set_xticklabels(method_names, rotation=10)
ax3.set_ylabel("Cost (units)")
ax3_twin.set_ylabel("Stability score")
ax3.set_title("Assignment summaries")
ax3.legend(loc="upper left")
ax3_twin.legend(loc="upper right")
ax3.grid(axis="y", alpha=0.3)

all_features, all_labels = create_feature_matrix(problem)
support_sizes = [1, 2, 3, 4, 5]
split_accuracies = []
for support_size in support_sizes:
    small_support, small_query = create_support_and_query_sets(
        problem, support_size=support_size
    )
    small_adapted_params = model.adapt_to_support_set(
        small_support["X"], small_support["y"], num_steps=5
    )
    small_metrics = evaluate_adapted_model(
        model, small_adapted_params, small_query
    )
    split_accuracies.append(small_metrics["accuracy"])

ax4.plot(
    support_sizes,
    np.array(split_accuracies) * 100.0,
    marker="o",
    linewidth=2,
    color="purple",
)
ax4.set_xlabel("Support-set size (examples)")
ax4.set_ylabel("Query pair-label accuracy (%)")
ax4.set_title("One random split per support size")
ax4.set_xticks(support_sizes)
ax4.set_ylim(0, 100)
ax4.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## Adaptation to a changed slot configuration

A second instance changes slot attributes and reuses the provided cost and stability matrices. This isolates the effect of changed slot features on the encoding and adaptation procedure; it does not recalculate operational costs or establish transfer to a genuinely new vessel. The pair labels consequently remain based on the same supplied cost–stability values. As in the first instance, support and query examples are disjoint within the new split.

In [ ]:
new_slots = [
    VesselSlot(1, 55, 3.2, 2, 14.0, "forward", 0.85, 0.92),
    VesselSlot(2, 36, 4.8, 3, 13.0, "midship", 0.88, 0.72),
    VesselSlot(3, 50, 3.2, 3, 17.0, "midship", 0.75, 0.82),
    VesselSlot(4, 66, 4.2, 3, 11.0, "aft", 0.92, 0.88),
]

new_problem = FSLProblem(
    containers=containers,
    slots=new_slots,
    cost_matrix=cost_matrix,
    stability_matrix=stability_matrix,
    bay_weight_limits=bay_weight_limits,
    feature_dim=feature_dim,
)

print("Changed slot capacities:")
for original_slot, changed_slot in zip(problem.slots, new_slots):
    percentage_change = (
        (changed_slot.weight_capacity - original_slot.weight_capacity)
        / original_slot.weight_capacity
        * 100.0
    )
    print(
        f"  Slot {changed_slot.id}: {changed_slot.weight_capacity} tons, "
        f"capacity change {percentage_change:+.1f}%"
    )

new_support_set, new_query_set = create_support_and_query_sets(
    new_problem, support_size=2
)
new_adapted_params = model.adapt_to_support_set(
    new_support_set["X"], new_support_set["y"], num_steps=5
)
new_metrics = evaluate_adapted_model(
    model, new_adapted_params, new_query_set
)
new_fsl_assignment = generate_assignment_from_predictions(
    model, new_adapted_params, new_problem
)
new_fsl_cost = float(np.sum(new_problem.cost_matrix * new_fsl_assignment))
new_fsl_stability = float(
    np.sum(new_problem.stability_matrix * new_fsl_assignment)
)

print("\nQuery pair-label metrics for the changed slot configuration:")
for metric_name, metric_value in new_metrics.items():
    print(f"  {metric_name.replace('_', ' ').title()}: {metric_value:.3f}")
print(f"Greedy feasible assignment cost: {new_fsl_cost:.2f} units")
print(f"Greedy feasible assignment stability: {new_fsl_stability:.2f}")
print(
    "These values describe one small random split and a cost/stability matrix "
    "that is held fixed across the two configurations."
)

## Interpretation and limitations

This example connects feature engineering, a feed-forward neural network, gradient-based support adaptation, query-set evaluation, and a score-based greedy assignment. The model's output is a pair score; it is not itself a complete assignment. The assignment stage adds one-to-one slot use and the stated weight, height, and bay-load checks, but its greedy choices do not guarantee minimum cost or even completion for every possible instance.

The proxy labels are generated from the same small set of supplied costs and stability values used to construct the example. They are not expert placement decisions or optimal-assignment labels. The small number of pairs, random initialization, random splits, and absence of training across multiple tasks prevent conclusions about reliable few-shot generalization, real-world placement accuracy, or superiority over optimization methods. The cost-only Hungarian reference is also not a constrained operational benchmark.

In a broader application, additional data and constraints could represent lashing compatibility, vessel stability, operational policies, and task-to-task variation. Those extensions require a suitable dataset and a clearly specified evaluation design; they are not implemented here.